In [ ]:
import os
import re
from whoosh.index import create_in, exists_in
from whoosh.fields import Schema, TEXT, ID
from whoosh.analysis import StandardAnalyzer

# daftar stopword bahasa Indonesia sederhana karna tidak boleh pakai NLTK/Sastrawi
STOPWORDS = {"yang", "di", "ke", "dari", "pada", "dalam", "untuk", "dan", "atau", "dengan", "ini", "itu", "adalah", "sebagai", "oleh"}

def preprocess_text(text):
    """
    Fungsi untuk membersihkan teks mentah sesuai standar minimum proyek UTS.
    Meliputi: Case folding, Tokenization (simpel), dan Stopword removal.
    """
    # 1. mengubah semua huruf menjadi huruf kecil
    text = text.lower()
    
    # regex
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    
    # 2. tokenisasi
    tokens = text.split()
    
    # 3. stopword removal
    clean_tokens = [word for word in tokens if word not in STOPWORDS]
    
    # penggabungan
    return " ".join(clean_tokens)

def setup_whoosh_index(index_dir="indexdir"):
    """
    Fungsi untuk membuat skema Whoosh dan menyiapkan direktori indeks.
    """
    # buat folder indeks
    if not os.path.exists(index_dir):
        os.mkdir(index_dir)
        
    # Mendefinisikan skema Whoosh:
    # - path: Untuk menyimpan lokasi asal file (ID, tidak dianalisis teksnya)
    # - source: Untuk menyimpan nama dataset (etd-usk, kompas, dll)
    # - content: Teks yang sudah di-preprocess dan siap dicari
    schema = Schema(
        path=ID(stored=True, unique=True),
        source=TEXT(stored=True),
        content=TEXT(analyzer=StandardAnalyzer(), stored=True)
    )
    
    # Membuat indeks baru
    ix = create_in(index_dir, schema)
    return ix

def load_and_index_datasets(base_dataset_path="dataset", index_dir="indexdir"):
    """
    Fungsi utama untuk membaca seluruh dataset dari folder, melakukan preprocessing,
    dan menyimpannya ke dalam indeks Whoosh.
    """
    print(f"Memulai proses Load & Index dari folder '{base_dataset_path}'...")
    
    # Whoosh index
    ix = setup_whoosh_index(index_dir)
    writer = ix.writer()
    
    # Dataset
    dataset_folders = ["etd-usk", "etd-ugm", "kompas", "tempo", "mojok"]
    
    doc_count = 0
    
    # Iterasi melalui setiap folder dataset
    for folder in dataset_folders:
        folder_path = os.path.join(base_dataset_path, folder)
        
        # Mengecek apakah folder dataset ada
        if not os.path.exists(folder_path):
            print(f"Peringatan: Folder {folder_path} tidak ditemukan. Lewati.")
            continue
            
        # Membaca setiap file teks dalam folder
        for filename in os.listdir(folder_path):
            file_path = os.path.join(folder_path, filename)
            
            # baca file
            if os.path.isfile(file_path):
                try:
                    # Membaca isi teks
                    with open(file_path, 'r', encoding='utf-8') as file:
                        raw_text = file.read()
                    
                    # Memanggil fungsi preprocessing
                    cleaned_text = preprocess_text(raw_text)
                    
                    # Menambahkan dokumen ke dalam Whoosh Index
                    writer.add_document(
                        path=file_path,
                        source=folder,
                        content=cleaned_text
                    )
                    doc_count += 1
                except Exception as e:
                    print(f"Gagal memproses file {file_path}: {e}")
                    
    # Menyimpan perubahan ke indeks
    writer.commit()
    print(f"Proses Indexing Selesai! Sebanyak {doc_count} dokumen berhasil diindeks ke dalam '{index_dir}'.")
    return ix



: 

In [ ]:
def main_menu():
    """
    CLI Interface.
    """
    while True:
        # tampilan
        print("\n=== INFORMATION RETRIEVAL SYSTEM ===")
        print("[1] Load & Index Dataset")
        print("[2] Search Query")
        print("[3] Exit")
        print("====================================")
        
        pilihan = input("Pilih menu (1/2/3): ")
        
        if pilihan == '1':
            # fungsi indexing 
            load_and_index_datasets(base_dataset_path="dataset", index_dir="indexdir")
        elif pilihan == '2':
            # input user
            query = input("Masukkan query pencarian: ")
            if query.strip():
                search_and_rank(query, index_dir="indexdir")
            else:
                print("Query tidak boleh kosong.")
        elif pilihan == '3':
            print("Keluar dari program. Terima kasih!")
            sys.exit(0)
        else:
            print("Pilihan tidak valid, silakan coba lagi.")